# Assignment 3: Support Vector Machine (SVM) Implementation
Student ID: 220119

This notebook implements **Support Vector Machine (SVM)** using scikit-learn on the Teens Mental Health Dataset.
SVM finds the optimal hyperplane that maximizes the margin between classes.

## Dataset
- **Teens Mental Health Dataset**
- Target: depression_label (0 = Not Depressed, 1 = Depressed)
- Features: age, gender, daily_social_media_hours, platform_usage, sleep_hours, screen_time_before_sleep, academic_performance, physical_activity, social_interaction_level, stress_level, anxiety_level, addiction_level

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix, classification_report)
from sklearn.inspection import DecisionBoundaryDisplay

## 1. Load the Dataset

In [2]:
# Load Teens Mental Health Dataset
import os
if os.path.exists("/home/ahsan/Downloads/Academic/3-2/AI and ML Lab/Lab 3/Teen_Mental_Health_Dataset.csv"):
    df = pd.read_csv("/home/ahsan/Downloads/Academic/3-2/AI and ML Lab/Lab 3/Teen_Mental_Health_Dataset.csv")
else:
    try:
        df = pd.read_csv("Teen_Mental_Health_Dataset.csv")
    except:
        url = "https://raw.githubusercontent.com/ahsanjust/Artificial-Intelligence-and-Machine-Learning-Lab/main/Lab%203/Teen_Mental_Health_Dataset.csv"
        df = pd.read_csv(url)

print("Dataset loaded successfully!")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Dataset loaded successfully!
Shape: (1200, 13)
Columns: ['age', 'gender', 'daily_social_media_hours', 'platform_usage', 'sleep_hours', 'screen_time_before_sleep', 'academic_performance', 'physical_activity', 'social_interaction_level', 'stress_level', 'anxiety_level', 'addiction_level', 'depression_label']


### 1.1 Exploratory Data Analysis

In [3]:
# First few rows
print("First 5 rows of the dataset:")
display(df.head())

# Dataset info
print("\nDataset Info:")
df.info()

# Statistical summary
print("\nStatistical Summary:")
display(df.describe())

First 5 rows of the dataset:


,age,gender,daily_social_media_hours,platform_usage,sleep_hours,screen_time_before_sleep,academic_performance,physical_activity,social_interaction_level,stress_level,anxiety_level,addiction_level,depression_label
0,14,male,7.9,Instagram,7.4,2.9,3.01,1.5,low,2,2,1,0
1,19,female,1.9,TikTok,8.0,2.9,3.22,0.8,high,8,1,10,0
2,17,female,1.3,Instagram,7.6,0.5,3.92,0.0,high,2,4,2,0
3,15,male,7.4,TikTok,6.9,1.6,3.48,0.8,medium,1,7,9,0
4,15,female,4.7,Both,4.9,3.0,2.37,1.4,medium,3,5,2,0



Dataset Info:
<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   age                       1200 non-null   int64  
 1   gender                    1200 non-null   str    
 2   daily_social_media_hours  1200 non-null   float64
 3   platform_usage            1200 non-null   str    
 4   sleep_hours               1200 non-null   float64
 5   screen_time_before_sleep  1200 non-null   float64
 6   academic_performance      1200 non-null   float64
 7   physical_activity         1200 non-null   float64
 8   social_interaction_level  1200 non-null   str    
 9   stress_level              1200 non-null   int64  
 10  anxiety_level             1200 non-null   int64  
 11  addiction_level           1200 non-null   int64  
 12  depression_label          1200 non-null   int64  
dtypes: float64(5), int64(5), str(3)
memory usage: 122.0 KB

Sta

,age,daily_social_media_hours,sleep_hours,screen_time_before_sleep,academic_performance,physical_activity,stress_level,anxiety_level,addiction_level,depression_label
count,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000
mean,15.928333,4.536667,6.449417,1.740333,2.990383,1.014500,5.445833,5.636667,5.565000,0.025833
std,2.021947,2.029599,1.442677,0.716660,0.576758,0.582185,2.903290,2.859453,2.830627,0.158704
min,13.000000,1.000000,4.000000,0.500000,2.000000,0.000000,1.000000,1.000000,1.000000,0.000000
25%,14.000000,2.800000,5.200000,1.100000,2.500000,0.500000,3.000000,3.000000,3.000000,0.000000
50%,16.000000,4.500000,6.500000,1.800000,2.990000,1.000000,5.000000,6.000000,6.000000,0.000000
75%,18.000000,6.300000,7.600000,2.400000,3.480000,1.500000,8.000000,8.000000,8.000000,0.000000
max,19.000000,8.000000,9.000000,3.000000,4.000000,2.000000,10.000000,10.000000,10.000000,1.000000


### 1.2 Target Variable Analysis

In [4]:
# Target variable distribution
print("Depression Label Distribution:")
print(df["depression_label"].value_counts())
print(f"\nClass 0 (Not Depressed): {(df['depression_label'] == 0).sum()} ({(df['depression_label'] == 0).mean()*100:.1f}%)")
print(f"Class 1 (Depressed):      {(df['depression_label'] == 1).sum()} ({(df['depression_label'] == 1).mean()*100:.1f}%)")

plt.figure(figsize=(6, 4))
sns.countplot(x="depression_label", data=df, palette=["#4CAF50", "#F44336"])
plt.xticks([0, 1], ["Not Depressed", "Depressed"])
plt.title("Distribution of Depression Label (Target)")
plt.tight_layout()
os.makedirs('screenshots', exist_ok=True)
plt.savefig('screenshots/Class_Distribution.png', dpi=150, bbox_inches='tight')
plt.show()

Depression Label Distribution:
depression_label
0    1169
1      31
Name: count, dtype: int64

Class 0 (Not Depressed): 1169 (97.4%)
Class 1 (Depressed):      31 (2.6%)


/tmp/ipykernel_14093/1647291665.py:8: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.countplot(x="depression_label", data=df, palette=["#4CAF50", "#F44336"])


/tmp/ipykernel_14093/1647291665.py:14: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


### 1.3 Correlation Analysis

In [5]:
# Encode categorical features for correlation
df_corr = df.copy()
for col in ["gender", "platform_usage", "social_interaction_level"]:
    df_corr[col] = LabelEncoder().fit_transform(df_corr[col])

# Correlation heatmap
plt.figure(figsize=(10, 8))
corr = df_corr.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True, linewidths=0.5)
plt.title("Feature Correlation Matrix", fontsize=14)
plt.tight_layout()
plt.savefig('screenshots/Correlation_Heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Top correlations with target
target_corr = corr["depression_label"].drop("depression_label").sort_values(ascending=False)
print("\nTop features correlated with depression_label:")
print(target_corr)


Top features correlated with depression_label:
daily_social_media_hours    0.175201
stress_level                0.170474
anxiety_level               0.169566
platform_usage              0.018264
age                         0.010973
social_interaction_level    0.005110
academic_performance        0.001441
addiction_level            -0.013952
screen_time_before_sleep   -0.016502
physical_activity          -0.017598
gender                     -0.019836
sleep_hours                -0.190630
Name: depression_label, dtype: float64


/tmp/ipykernel_14093/3948625478.py:13: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


### 1.4 Feature Distributions

In [6]:
num_features = ["age", "daily_social_media_hours", "sleep_hours", "screen_time_before_sleep",
                "physical_activity", "stress_level", "anxiety_level", "addiction_level"]

fig, axes = plt.subplots(4, 2, figsize=(14, 12))
axes = axes.flatten()
for i, feat in enumerate(num_features):
    sns.histplot(df[feat], bins=20, kde=True, ax=axes[i], color="purple")
    axes[i].set_title(feat.replace("_", " ").title())
plt.tight_layout()
plt.savefig('screenshots/Feature_Distributions.png', dpi=150, bbox_inches='tight')
plt.show()

/tmp/ipykernel_14093/2871404413.py:11: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## 2. Data Preprocessing

### 2.1 Check for Missing Values

In [7]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nTotal missing values: {df.isnull().sum().sum()}")

Missing values per column:
age                         0
gender                      0
daily_social_media_hours    0
platform_usage              0
sleep_hours                 0
screen_time_before_sleep    0
academic_performance        0
physical_activity           0
social_interaction_level    0
stress_level                0
anxiety_level               0
addiction_level             0
depression_label            0
dtype: int64

Total missing values: 0


### 2.2 Handle Categorical Variables
Label Encoding is used here because:
- `gender`, `platform_usage`, and `social_interaction_level` are ordinal or binary categories
- For SVM, label encoding with standardization works well and keeps feature dimensions low
- One-hot encoding would increase dimensionality, potentially making the SVM margin computation more expensive

In [8]:
# Handle categorical variables - encode them numerically
df_encoded = df.copy()

label_encoders = {}
categorical_columns = ['gender', 'platform_usage', 'social_interaction_level']

for col in categorical_columns:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])
    label_encoders[col] = le

print("Categorical encoding completed!")
for col, le in label_encoders.items():
    print(f"{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

Categorical encoding completed!
gender: {'female': 0, 'male': 1}
platform_usage: {'Both': 0, 'Instagram': 1, 'TikTok': 2}
social_interaction_level: {'high': 0, 'low': 1, 'medium': 2}


### 2.3 Train-Validation-Test Split
Stratified split ensures both classes are proportionally represented in all subsets.

In [9]:
# Prepare features and target
X = df_encoded.drop('depression_label', axis=1)
y = df_encoded['depression_label']

# Stratified split: 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)
# Split temp: 50% validation, 50% test (15% each of total)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f"Training samples:   {X_train.shape[0]} ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Validation samples: {X_val.shape[0]} ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Test samples:       {X_test.shape[0]} ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"\nTraining set class distribution:")
print(y_train.value_counts().to_dict())

Training samples:   840 (70.0%)
Validation samples: 180 (15.0%)
Test samples:       180 (15.0%)

Training set class distribution:
{0: 818, 1: 22}


### 2.4 Feature Scaling
Standardization is essential for SVM because:
- SVM maximizes the margin between classes, which depends on distance computations
- Features with larger magnitudes would dominate the margin calculation
- The regularization parameter C is equally applied across all features only when they're on the same scale

In [10]:
# Standardize features (mandatory for SVM)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed!")
print(f"X_train_scaled mean: {X_train_scaled.mean():.6f}")
print(f"X_train_scaled std:  {X_train_scaled.std():.6f}")

Feature scaling completed!
X_train_scaled mean: 0.000000
X_train_scaled std:  1.000000


## 3. SVM Model Training with Hyperparameter Tuning
We compare **Linear Kernel** vs **RBF Kernel** with hyperparameter tuning:
- **C**: Regularization parameter (trade-off between margin size and misclassification)
- **kernel**: linear vs rbf
- **gamma** (for RBF): Kernel coefficient that controls the influence of each training sample

`class_weight='balanced'` is used to handle the severe class imbalance automatically.

In [11]:
# Define parameter grid for SVM
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto']
}

# Initialize SVM with class_weight='balanced' for imbalance
svm_base = SVC(class_weight='balanced', probability=True, random_state=42)

# Grid search with stratified cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search_svm = GridSearchCV(
    estimator=svm_base,
    param_grid=param_grid,
    scoring='f1',
    cv=cv,
    n_jobs=-1,
    verbose=1
)

# Train
print("Running GridSearchCV for SVM hyperparameter tuning...")
grid_search_svm.fit(X_train_scaled, y_train)

print(f"\nBest parameters: {grid_search_svm.best_params_}")
print(f"Best cross-validation F1-score: {grid_search_svm.best_score_:.4f}")

best_svm = grid_search_svm.best_estimator_

Running GridSearchCV for SVM hyperparameter tuning...
Fitting 5 folds for each of 20 candidates, totalling 100 fits



Best parameters: {'C': 1, 'gamma': 'scale', 'kernel': 'rbf'}
Best cross-validation F1-score: 0.5229


### 3.1 Linear vs RBF Kernel Comparison

In [12]:
# Train separate models for comparison
svm_linear = SVC(kernel='linear', C=1, class_weight='balanced', probability=True, random_state=42)
svm_rbf = SVC(kernel='rbf', C=1, gamma='scale', class_weight='balanced', probability=True, random_state=42)

svm_linear.fit(X_train_scaled, y_train)
svm_rbf.fit(X_train_scaled, y_train)

# Evaluate on validation set
for name, model in [('Linear Kernel', svm_linear), ('RBF Kernel', svm_rbf)]:
    y_val_pred = model.predict(X_val_scaled)
    y_val_prob = model.predict_proba(X_val_scaled)[:, 1]
    f1_val = f1_score(y_val, y_val_pred)
    auc_val = roc_auc_score(y_val, y_val_prob)
    print(f"{name} -> Validation F1: {f1_val:.4f}, AUC: {auc_val:.4f}")

Linear Kernel -> Validation F1: 0.5000, AUC: 0.9726
RBF Kernel -> Validation F1: 0.6667, AUC: 0.9909


## 4. Model Evaluation

In [13]:
# Predict on test set with best model
y_pred = best_svm.predict(X_test_scaled)
y_prob = best_svm.predict_proba(X_test_scaled)[:, 1]

# Calculate metrics
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print("=" * 55)
print("SVM MODEL EVALUATION")
print("=" * 55)
print(f"Accuracy:     {acc:.4f} ({acc*100:.2f}%)")
print(f"Precision:    {prec:.4f}")
print(f"Recall:       {rec:.4f}")
print(f"F1-Score:     {f1:.4f}")
print(f"AUC-ROC:      {auc:.4f}")
print(f"Best Kernel:  {grid_search_svm.best_params_['kernel']}")
print(f"Best C:       {grid_search_svm.best_params_['C']}")
print(f"Best Gamma:   {grid_search_svm.best_params_['gamma']}")
print("=" * 55)

SVM MODEL EVALUATION
Accuracy:     0.9778 (97.78%)
Precision:    0.5000
Recall:       0.2500
F1-Score:     0.3333
AUC-ROC:      0.9631
Best Kernel:  rbf
Best C:       1
Best Gamma:   scale


### 4.1 Confusion Matrix

In [14]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Depressed (0)', 'Depressed (1)'],
            yticklabels=['Not Depressed (0)', 'Depressed (1)'],
            annot_kws={'size': 14})
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.title('Confusion Matrix - SVM', fontsize=14)
plt.tight_layout()
plt.savefig('screenshots/Confusion_Matrix.png', dpi=150, bbox_inches='tight')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Negatives (Not Depressed correct):  {tn}")
print(f"False Positives (Depressed wrong):       {fp}")
print(f"False Negatives (Not Depressed wrong):   {fn}")
print(f"True Positives (Depressed correct):      {tp}")

True Negatives (Not Depressed correct):  175
False Positives (Depressed wrong):       1
False Negatives (Not Depressed wrong):   3
True Positives (Depressed correct):      1


/tmp/ipykernel_14093/2778227189.py:13: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


### 4.2 ROC Curve

In [15]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, 'b-', linewidth=2, label=f'SVM (AUC = {auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Random Classifier')
plt.fill_between(fpr, tpr, alpha=0.2)
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve - SVM', fontsize=14)
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('screenshots/ROC_Curve.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"AUC = {auc:.4f}")
print("AUC of 1.0 = Perfect classifier")
print("AUC of 0.5 = Random classifier")

AUC = 0.9631
AUC of 1.0 = Perfect classifier
AUC of 0.5 = Random classifier


/tmp/ipykernel_14093/3136158281.py:14: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


### 4.3 2D Classification Plot (Decision Boundary)

In [16]:
# For 2D visualization, select the two most important features
feature_names = list(X.columns)
top_features = ['anxiety_level', 'stress_level']
idx1 = feature_names.index(top_features[0])
idx2 = feature_names.index(top_features[1])

# Train SVM on just these 2 features
svm_2d = SVC(
    kernel=grid_search_svm.best_params_['kernel'],
    C=grid_search_svm.best_params_['C'],
    gamma=grid_search_svm.best_params_['gamma'],
    class_weight='balanced',
    probability=True,
    random_state=42
)
svm_2d.fit(X_train_scaled[:, [idx1, idx2]], y_train)

# Plot decision boundary
plt.figure(figsize=(10, 8))
DecisionBoundaryDisplay.from_estimator(
    svm_2d, X_train_scaled[:, [idx1, idx2]],
    response_method='predict',
    cmap='coolwarm',
    alpha=0.6,
    grid_resolution=200
)
scatter = plt.scatter(
    X_test_scaled[:, idx1], X_test_scaled[:, idx2],
    c=y_test, cmap='coolwarm', edgecolors='k', s=50
)
plt.xlabel(f'{top_features[0]} (standardized)', fontsize=12)
plt.ylabel(f'{top_features[1]} (standardized)', fontsize=12)
plt.title(f'SVM Decision Boundary (kernel={grid_search_svm.best_params_["kernel"]}, C={grid_search_svm.best_params_["C"]})', fontsize=14)
plt.tight_layout()
plt.savefig('screenshots/Decision_Boundary.png', dpi=150, bbox_inches='tight')
plt.show()

/tmp/ipykernel_14093/406028216.py:36: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


### 4.4 Sample Predictions with Probabilities

In [17]:
y_prob_full = best_svm.predict_proba(X_test_scaled)[:, 1]

sample_df = pd.DataFrame({
    'Actual': y_test[:15].values,
    'Predicted': y_pred[:15],
    'Probability (Depressed)': y_prob_full[:15]
})
sample_df['Actual Label'] = sample_df['Actual'].map({0: 'Not Depressed', 1: 'Depressed'})
sample_df['Predicted Label'] = sample_df['Predicted'].map({0: 'Not Depressed', 1: 'Depressed'})

print("Sample Predictions with Probability Scores:")
display(sample_df[['Actual Label', 'Predicted Label', 'Probability (Depressed)']])

correct = (y_pred == y_test.values).sum()
print(f"\nCorrect predictions: {correct}/{len(y_test)} ({correct/len(y_test)*100:.1f}%)")

Sample Predictions with Probability Scores:


,Actual Label,Predicted Label,Probability (Depressed)
0,Not Depressed,Not Depressed,0.000550
1,Not Depressed,Not Depressed,0.001768
2,Not Depressed,Not Depressed,0.009371
3,Not Depressed,Not Depressed,0.000288
4,Not Depressed,Not Depressed,0.000783
5,Not Depressed,Not Depressed,0.000146
6,Not Depressed,Not Depressed,0.000191
7,Not Depressed,Not Depressed,0.000100
8,Not Depressed,Not Depressed,0.000165
9,Not Depressed,Not Depressed,0.000352



Correct predictions: 176/180 (97.8%)


## 5. Analytical Comparison: KNN vs SVM

### Model Characteristics
| Aspect | KNN | SVM |
|--------|-----|-----|
| **Type** | Lazy (instance-based) | Eager (margin-based) |
| **Training Speed** | O(1) — no training | O(n²) to O(n³) |
| **Inference Speed** | O(n) — scans all points | O(k) — depends on support vectors |
| **Memory** | Stores all training data | Stores only support vectors |
| **Decision Boundary** | Non-linear, piecewise | Linear or kernel-based |
| **Hyperparameters** | k, weights, metric | C, kernel, gamma |

### Performance on This Dataset
*After running both notebooks, fill in the actual KNN metrics below.*
| Metric | KNN | SVM |
|--------|-----|-----|
| Accuracy | 97.78% | 97.78% |
| Precision | 0.0000 | 0.5000 |
| Recall | 0.0000 | 0.2500 |
| F1-Score | 0.0000 | 0.3333 |
| AUC-ROC | 0.6080 | 0.9631 |

### Key Takeaways
1. **SVM significantly outperforms KNN** on this imbalanced dataset — SVM achieves AUC=0.96 vs KNN's 0.61, and F1=0.33 vs KNN's 0.0.
2. **KNN fails to detect the minority class** — precision and recall are both 0, meaning KNN predicts "Not Depressed" for every test sample.
3. **SVM with RBF kernel** and `class_weight='balanced'` captures the non-linear decision boundary better and handles the 2.5% positive class imbalance effectively.
4. **Feature Scaling** is critical for both algorithms since both rely on distance computations.
5. KNN's lazy nature makes it faster to "train" but slower to predict, while SVM requires more training time but makes faster predictions.

In [18]:
# Store SVM metrics for comparison
svm_metrics = {
    'acc': acc, 'prec': prec, 'rec': rec, 'f1': f1, 'auc': auc
}
print("SVM metrics captured for comparison.")
print("Save KNN metrics from the KNN notebook and compare manually.")
print("\nSVM Summary:")
print(f"  Best kernel: {grid_search_svm.best_params_['kernel']}")
print(f"  Best C:      {grid_search_svm.best_params_['C']}")
print(f"  Best gamma:  {grid_search_svm.best_params_['gamma']}")
print(f"  Test F1:     {f1:.4f}")
print(f"  Test AUC:    {auc:.4f}")

SVM metrics captured for comparison.
Save KNN metrics from the KNN notebook and compare manually.

SVM Summary:
  Best kernel: rbf
  Best C:      1
  Best gamma:  scale
  Test F1:     0.3333
  Test AUC:    0.9631
